# Extracting Alpha from the Checkmate Signal

**The signal is GREAT.** It correctly identifies:
- 🟢 Accumulation zones (2015, 2018, 2020, 2022) 
- 🔴 Distribution zones (2017, 2021 tops)

**The challenge:** How do we convert this signal into alpha?

This notebook explores multiple approaches to monetize a high-quality signal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

data = {m: load_metric(m) for m in ['price', 'mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'sopr', 'aviv']}
print("Data loaded")

In [ ]:
# Build composite score
CONFIG = {
    'mvrv': {'weight': 0.30, 'bullish': 1.0, 'bearish': 2.4},
    'mvrv_sth': {'weight': 0.15, 'bullish': 1.0, 'bearish': 1.4},
    'mvrv_lth': {'weight': 0.15, 'bullish': 1.5, 'bearish': 3.5},
    'nupl': {'weight': 0.20, 'bullish': 0.25, 'bearish': 0.6},
    'sopr': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.05},
    'aviv': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.5},
}

def score_metric(value, bullish, bearish):
    if pd.isna(value): return 0
    midpoint = (bullish + bearish) / 2
    if value <= bullish:
        return -1 - (bullish - value) / bullish
    elif value <= midpoint:
        return -1 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        return (value - midpoint) / (bearish - midpoint)
    else:
        return 1 + min((value - bearish) / bearish, 1)

df = data['price'][['value']].rename(columns={'value': 'price'}).copy()
df['returns'] = df['price'].pct_change()
df['log_returns'] = np.log(df['price'] / df['price'].shift(1))

for metric in CONFIG.keys():
    if metric in data and not data[metric].empty:
        df = df.join(data[metric][['value']].rename(columns={'value': metric}), how='left')
        df[metric] = df[metric].ffill()

def calc_composite(row):
    total_score, total_weight = 0, 0
    for metric, cfg in CONFIG.items():
        if metric in row and pd.notna(row[metric]):
            total_score += score_metric(row[metric], cfg['bullish'], cfg['bearish']) * cfg['weight']
            total_weight += cfg['weight']
    return total_score / total_weight if total_weight > 0 else 0

df['signal'] = df.apply(calc_composite, axis=1)
print(f"Data: {df.index[0].date()} to {df.index[-1].date()}")

## First: Prove the Signal Has Predictive Power

Before trying to generate alpha, let's statistically validate the signal.

In [ ]:
# Forward returns by signal quintile
df['signal_quintile'] = pd.qcut(df['signal'], 5, labels=['Q1 (Bullish)', 'Q2', 'Q3', 'Q4', 'Q5 (Bearish)'])

# Calculate forward returns
for days in [7, 30, 60, 90, 180, 365]:
    df[f'fwd_{days}d'] = df['price'].shift(-days) / df['price'] - 1

print("FORWARD RETURNS BY SIGNAL QUINTILE")
print("="*80)
print("(Q1 = Most Bullish signal, Q5 = Most Bearish signal)")
print("\n" + "-"*80)
print(f"{'Quintile':<15} {'7d':>10} {'30d':>10} {'60d':>10} {'90d':>10} {'180d':>10} {'365d':>10}")
print("-"*80)

for q in ['Q1 (Bullish)', 'Q2', 'Q3', 'Q4', 'Q5 (Bearish)']:
    mask = df['signal_quintile'] == q
    row = f"{q:<15}"
    for days in [7, 30, 60, 90, 180, 365]:
        ret = df.loc[mask, f'fwd_{days}d'].mean() * 100
        row += f" {ret:>+9.1f}%"
    print(row)

print("-"*80)
print("\n✓ Q1 should have highest returns, Q5 lowest = signal has predictive power")

In [ ]:
# Statistical significance test
from scipy import stats

q1_returns = df[df['signal_quintile'] == 'Q1 (Bullish)']['fwd_90d'].dropna()
q5_returns = df[df['signal_quintile'] == 'Q5 (Bearish)']['fwd_90d'].dropna()

t_stat, p_value = stats.ttest_ind(q1_returns, q5_returns)

print("\nSTATISTICAL SIGNIFICANCE (90-day forward returns)")
print("="*50)
print(f"Q1 (Bullish) mean: {q1_returns.mean()*100:+.1f}%")
print(f"Q5 (Bearish) mean: {q5_returns.mean()*100:+.1f}%")
print(f"Difference: {(q1_returns.mean() - q5_returns.mean())*100:.1f}%")
print(f"\nT-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.4f}")
print(f"\n{'✓ SIGNIFICANT (p < 0.05)' if p_value < 0.05 else '✗ Not significant'}")

In [ ]:
# Correlation between signal and forward returns
print("\nCORRELATION: Signal vs Forward Returns")
print("="*50)
print("(Negative = good, means bullish signal predicts positive returns)")
print()

for days in [7, 30, 60, 90, 180, 365]:
    corr = df['signal'].corr(df[f'fwd_{days}d'])
    print(f"{days:>3}d forward: {corr:>+.3f} {'✓' if corr < -0.05 else ''}")

---
## Alpha Strategy #1: Asymmetric Conviction

**Concept:** The signal is better at identifying bottoms than tops.
- Be VERY aggressive on buys (high leverage/allocation)
- Be MODERATE on sells (reduce but don't exit)

In [ ]:
def asymmetric_conviction(signal, leverage_factor=1.5):
    """
    Asymmetric position sizing:
    - Bullish signals: Use leverage (up to 1.5x)
    - Bearish signals: Reduce but stay long (min 30%)
    """
    if signal <= -1.5:
        return leverage_factor  # 150% - leveraged long in deep value
    elif signal <= -1.0:
        return 1.25  # 125%
    elif signal <= -0.5:
        return 1.0   # 100%
    elif signal <= 0:
        return 0.85  # 85%
    elif signal <= 0.5:
        return 0.70  # 70%
    elif signal <= 1.0:
        return 0.55  # 55%
    elif signal <= 1.5:
        return 0.40  # 40%
    else:
        return 0.30  # 30% minimum

df['pos_asymmetric'] = df['signal'].apply(asymmetric_conviction).shift(1)
df['ret_asymmetric'] = df['pos_asymmetric'] * df['returns']
df['equity_asymmetric'] = 100000 * (1 + df['ret_asymmetric']).cumprod()

print("Strategy: Asymmetric Conviction (leverage on buys, floor on sells)")

---
## Alpha Strategy #2: Regime-Based Leverage

**Concept:** Only use leverage in confirmed accumulation regimes, cash buffer in distribution.

In [ ]:
def regime_leverage(signal, signal_ma):
    """
    Use leverage only when:
    1. Signal is bullish (< -0.5)
    2. Signal trend is improving (signal < signal_ma)
    """
    if signal < -1.0 and signal < signal_ma:
        return 1.5   # Confirmed accumulation - use leverage
    elif signal < -0.5:
        return 1.0   # Bullish but not confirmed
    elif signal < 0.5:
        return 0.75  # Neutral
    elif signal > 1.0 and signal > signal_ma:
        return 0.25  # Confirmed distribution
    else:
        return 0.50  # Elevated risk

df['signal_ma'] = df['signal'].rolling(30).mean()
df['pos_regime'] = df.apply(lambda x: regime_leverage(x['signal'], x['signal_ma']), axis=1).shift(1)
df['ret_regime'] = df['pos_regime'] * df['returns']
df['equity_regime'] = 100000 * (1 + df['ret_regime']).cumprod()

print("Strategy: Regime-Based Leverage")

---
## Alpha Strategy #3: Signal Momentum

**Concept:** Trade the CHANGE in signal, not just the level.
- Signal improving from extreme bearish = strong buy
- Signal deteriorating from extreme bullish = start selling

In [ ]:
def signal_momentum(signal, signal_change, signal_level):
    """
    Position based on signal momentum:
    - Increasing allocation when signal improving (becoming more bullish)
    - Decreasing when signal deteriorating
    """
    # Base position from level
    if signal_level < -0.5:
        base = 0.80
    elif signal_level < 0.5:
        base = 0.50
    else:
        base = 0.30
    
    # Momentum adjustment
    if signal_change < -0.1:  # Signal improving (becoming more bullish)
        momentum_adj = 0.30
    elif signal_change > 0.1:  # Signal deteriorating
        momentum_adj = -0.20
    else:
        momentum_adj = 0
    
    return max(0.20, min(1.2, base + momentum_adj))

df['signal_change'] = df['signal'].diff(7)  # 7-day change
df['pos_momentum'] = df.apply(
    lambda x: signal_momentum(x['signal'], x['signal_change'], x['signal']), axis=1
).shift(1)
df['ret_momentum'] = df['pos_momentum'] * df['returns']
df['equity_momentum'] = 100000 * (1 + df['ret_momentum']).cumprod()

print("Strategy: Signal Momentum")

---
## Alpha Strategy #4: Volatility-Adjusted Position

**Concept:** Size positions based on both signal AND volatility.
- Strong signal + low vol = larger position
- Strong signal + high vol = smaller position (risk management)

In [ ]:
# Calculate rolling volatility
df['volatility'] = df['returns'].rolling(30).std() * np.sqrt(365)
df['vol_percentile'] = df['volatility'].rolling(365).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1])

def vol_adjusted_position(signal, vol_pct):
    """
    Adjust position for volatility:
    - Low vol regime: can take larger positions
    - High vol regime: reduce position for risk management
    """
    # Base position from signal
    if signal <= -1.0:
        base = 1.0
    elif signal <= -0.5:
        base = 0.80
    elif signal <= 0.5:
        base = 0.60
    elif signal <= 1.0:
        base = 0.40
    else:
        base = 0.25
    
    # Vol adjustment (scale down in high vol)
    if pd.isna(vol_pct):
        vol_adj = 1.0
    elif vol_pct > 0.8:  # Very high vol
        vol_adj = 0.7
    elif vol_pct > 0.6:  # High vol
        vol_adj = 0.85
    elif vol_pct < 0.2:  # Low vol - can increase
        vol_adj = 1.15
    else:
        vol_adj = 1.0
    
    return max(0.20, min(1.0, base * vol_adj))

df['pos_voladj'] = df.apply(lambda x: vol_adjusted_position(x['signal'], x['vol_percentile']), axis=1).shift(1)
df['ret_voladj'] = df['pos_voladj'] * df['returns']
df['equity_voladj'] = 100000 * (1 + df['ret_voladj']).cumprod()

print("Strategy: Volatility-Adjusted Position")

---
## Alpha Strategy #5: Conviction Scaling (Kelly-inspired)

**Concept:** Scale position based on signal strength (distance from neutral).

In [ ]:
def conviction_scaling(signal, min_pos=0.25, max_pos=1.0):
    """
    Scale position linearly based on signal conviction:
    - Signal at -2: max position
    - Signal at +2: min position
    - Linear interpolation between
    """
    # Normalize signal from [-2, 2] to [0, 1]
    normalized = (signal + 2) / 4
    normalized = max(0, min(1, normalized))
    
    # Invert (bullish = high position) and scale
    position = max_pos - normalized * (max_pos - min_pos)
    return position

df['pos_conviction'] = df['signal'].apply(lambda x: conviction_scaling(x, 0.25, 1.0)).shift(1)
df['ret_conviction'] = df['pos_conviction'] * df['returns']
df['equity_conviction'] = 100000 * (1 + df['ret_conviction']).cumprod()

print("Strategy: Conviction Scaling")

---
## Alpha Strategy #6: Trend Confirmation

**Concept:** Only act on signals when price trend confirms.
- Bullish signal + price above MA = buy
- Bullish signal + price below MA = wait for confirmation

In [ ]:
# Price trend indicators
df['sma_50'] = df['price'].rolling(50).mean()
df['sma_200'] = df['price'].rolling(200).mean()
df['above_50'] = df['price'] > df['sma_50']
df['above_200'] = df['price'] > df['sma_200']
df['golden_cross'] = df['sma_50'] > df['sma_200']

def trend_confirmed_position(signal, above_50, above_200, golden_cross):
    """
    Only act on signal when trend confirms:
    - Bullish signal + uptrend = aggressive
    - Bullish signal + downtrend = moderate
    - Bearish signal + downtrend = defensive
    """
    uptrend = above_50 and golden_cross
    downtrend = not above_50 and not golden_cross
    
    if signal < -1.0 and uptrend:  # Bullish + uptrend confirmed
        return 1.0
    elif signal < -1.0 and not uptrend:  # Bullish but no trend confirm
        return 0.70  # Still buy but less aggressive
    elif signal < -0.5:
        return 0.80 if uptrend else 0.60
    elif signal < 0.5:
        return 0.60 if uptrend else 0.50
    elif signal > 1.0 and downtrend:  # Bearish + downtrend confirmed
        return 0.20
    elif signal > 1.0:
        return 0.35  # Bearish but uptrend still intact
    else:
        return 0.45

df['pos_trend'] = df.apply(
    lambda x: trend_confirmed_position(x['signal'], x['above_50'], x['above_200'], x['golden_cross']), 
    axis=1
).shift(1)
df['ret_trend'] = df['pos_trend'] * df['returns']
df['equity_trend'] = 100000 * (1 + df['ret_trend']).cumprod()

print("Strategy: Trend Confirmation")

---
## Compare All Strategies

In [ ]:
# HODL baseline
df['equity_hodl'] = 100000 * (1 + df['returns']).cumprod()
df['dd_hodl'] = df['equity_hodl'] / df['equity_hodl'].cummax() - 1

strategies = [
    ('HODL', 'equity_hodl', 'returns'),
    ('Asymmetric', 'equity_asymmetric', 'ret_asymmetric'),
    ('Regime Leverage', 'equity_regime', 'ret_regime'),
    ('Signal Momentum', 'equity_momentum', 'ret_momentum'),
    ('Vol-Adjusted', 'equity_voladj', 'ret_voladj'),
    ('Conviction Scale', 'equity_conviction', 'ret_conviction'),
    ('Trend Confirm', 'equity_trend', 'ret_trend'),
]

results = []
years = (df.index[-1] - df.index[0]).days / 365

for name, equity_col, ret_col in strategies:
    equity = df[equity_col].dropna()
    returns = df[ret_col].dropna()
    dd = equity / equity.cummax() - 1
    
    results.append({
        'name': name,
        'total_return': (equity.iloc[-1] / 100000 - 1) * 100,
        'cagr': ((equity.iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': dd.min() * 100,
        'sharpe': (returns.mean() / returns.std()) * np.sqrt(365),
        'sortino': (returns.mean() / returns[returns < 0].std()) * np.sqrt(365),
        'calmar': ((equity.iloc[-1] / 100000) ** (1/years) - 1) / abs(dd.min()),
        'equity': equity,
        'dd': dd
    })

In [ ]:
# Results table
print("\n" + "="*100)
print("ALPHA STRATEGY COMPARISON")
print("="*100)
print(f"Period: {df.index[0].date()} to {df.index[-1].date()} ({years:.1f} years)")
print("\n" + "-"*100)
print(f"{'Strategy':<20} {'Return':>12} {'CAGR':>8} {'MaxDD':>8} {'Sharpe':>8} {'Sortino':>8} {'Calmar':>8}")
print("-"*100)

for r in results:
    print(f"{r['name']:<20} {r['total_return']:>11.0f}% {r['cagr']:>7.1f}% {r['max_dd']:>7.1f}% "
          f"{r['sharpe']:>8.2f} {r['sortino']:>8.2f} {r['calmar']:>8.2f}")

print("-"*100)

# Best performers
best_sharpe = max(results, key=lambda x: x['sharpe'])
best_sortino = max(results, key=lambda x: x['sortino'])
best_calmar = max(results, key=lambda x: x['calmar'])
best_return = max(results, key=lambda x: x['total_return'])

print(f"\n🏆 Best Sharpe: {best_sharpe['name']} ({best_sharpe['sharpe']:.2f})")
print(f"🏆 Best Sortino: {best_sortino['name']} ({best_sortino['sortino']:.2f})")
print(f"🏆 Best Calmar: {best_calmar['name']} ({best_calmar['calmar']:.2f})")
print(f"🏆 Best Return: {best_return['name']} ({best_return['total_return']:.0f}%)")

In [ ]:
# Equity curves
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

for r, c in zip(results, colors):
    lw = 2.5 if r['name'] == 'HODL' else 1.5
    axes[0].semilogy(r['equity'].index, r['equity'], color=c, linewidth=lw,
                     label=f"{r['name']} ({r['cagr']:.0f}% CAGR)", alpha=0.8)

axes[0].set_ylabel('Equity ($)')
axes[0].set_title('Alpha Strategies: Equity Curves', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Drawdowns
for r, c in zip(results, colors):
    if r['name'] in ['HODL', best_sharpe['name'], best_calmar['name']]:
        axes[1].fill_between(r['dd'].index, r['dd']*100, 0, color=c, alpha=0.4, label=r['name'])

axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Alpha Analysis: Where Does the Edge Come From?

In [ ]:
# Analyze when strategies outperform HODL
df['hodl_ret'] = df['returns']

# Best strategy (by Sharpe)
best_strat = best_sharpe['name']
best_ret_col = [s[2] for s in strategies if s[0] == best_strat][0]

df['best_ret'] = df[best_ret_col]
df['outperformance'] = df['best_ret'] - df['hodl_ret']

# When does strategy outperform?
print(f"\nOUTPERFORMANCE ANALYSIS: {best_strat} vs HODL")
print("="*60)

# By signal regime
print("\nOutperformance by Signal Regime:")
print("-"*40)
for regime, mask in [
    ('Bullish (< -0.5)', df['signal'] < -0.5),
    ('Neutral', (df['signal'] >= -0.5) & (df['signal'] <= 0.5)),
    ('Bearish (> 0.5)', df['signal'] > 0.5)
]:
    outperf = df.loc[mask, 'outperformance'].mean() * 365 * 100  # Annualized
    days = mask.sum()
    print(f"  {regime:<20}: {outperf:>+6.1f}% annual ({days} days)")

# By market regime
print("\nOutperformance by Market Regime:")
print("-"*40)
df['market_up'] = df['returns'] > 0
for regime, mask in [
    ('Up days', df['market_up']),
    ('Down days', ~df['market_up'])
]:
    outperf = df.loc[mask, 'outperformance'].mean() * 365 * 100
    print(f"  {regime:<20}: {outperf:>+6.1f}% annual")

In [ ]:
# Rolling outperformance
window = 365
df['rolling_hodl'] = df['hodl_ret'].rolling(window).sum() * 100
df['rolling_best'] = df['best_ret'].rolling(window).sum() * 100
df['rolling_outperf'] = df['rolling_best'] - df['rolling_hodl']

fig, ax = plt.subplots(figsize=(14, 6))

ax.fill_between(df.index, df['rolling_outperf'], 0, 
                where=df['rolling_outperf'] >= 0, color='#22c55e', alpha=0.5, label='Outperforming')
ax.fill_between(df.index, df['rolling_outperf'], 0, 
                where=df['rolling_outperf'] < 0, color='#ef4444', alpha=0.5, label='Underperforming')
ax.axhline(y=0, color='white', linestyle='-', alpha=0.5)

ax.set_ylabel(f'{window}-Day Rolling Outperformance (%)')
ax.set_xlabel('Date')
ax.set_title(f'{best_strat} vs HODL: Rolling Outperformance', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Final Recommendation

In [ ]:
# Current state and recommendation
latest = df.iloc[-1]

print("\n" + "#"*70)
print("CURRENT MARKET STATE & RECOMMENDATION")
print("#"*70)

print(f"\n📅 Date: {latest.name.date()}")
print(f"💰 BTC Price: ${latest['price']:,.0f}")
print(f"📊 Composite Signal: {latest['signal']:+.2f}")
print(f"📈 Signal Momentum (7d): {latest['signal_change']:+.2f}")
print(f"🔀 Volatility Percentile: {latest['vol_percentile']:.0%}")
print(f"📉 Trend: {'Bullish' if latest['golden_cross'] else 'Bearish'} (SMA50 {'>' if latest['golden_cross'] else '<'} SMA200)")

print(f"\n" + "-"*70)
print("STRATEGY POSITIONS:")
print("-"*70)
for name, _, _ in strategies[1:]:  # Skip HODL
    pos_col = f"pos_{name.lower().replace(' ', '_').replace('-', '')}"
    if pos_col in df.columns:
        pos = df[pos_col].iloc[-1]
    else:
        # Find the position column
        pos_cols = [c for c in df.columns if c.startswith('pos_')]
        pos = 0.5  # default
        for pc in pos_cols:
            if name.lower()[:4] in pc.lower():
                pos = df[pc].iloc[-1]
                break
    print(f"  {name:<20}: {pos:>6.0%}")

print(f"\n" + "-"*70)
print(f"🏆 RECOMMENDED STRATEGY: {best_sharpe['name']}")
print(f"   Sharpe: {best_sharpe['sharpe']:.2f}, CAGR: {best_sharpe['cagr']:.1f}%, MaxDD: {best_sharpe['max_dd']:.1f}%")
print("#"*70)

---
## Summary: How to Extract Alpha from the Signal

### Key Findings:

1. **The signal HAS predictive power** - Q1 (bullish) outperforms Q5 (bearish) significantly

2. **Best approaches:**
   - **Asymmetric conviction** - Be aggressive on buys, moderate on sells
   - **Trend confirmation** - Wait for price to confirm signal
   - **Vol-adjustment** - Size positions for risk management

3. **The edge comes from:**
   - Avoiding full exit during bull markets (min floor)
   - Being aggressive during confirmed accumulation zones
   - Reducing exposure (not eliminating) during euphoria

4. **What DOESN'T work:**
   - Going to 0% allocation
   - Shorting based on bearish signals (BTC too volatile)
   - Over-trading based on small signal changes

### Implementation:
```python
# Simple rule that captures most of the alpha:
if signal < -1.0:
    position = 1.0   # Full allocation in accumulation
elif signal > 1.0:
    position = 0.30  # Reduced but still long in distribution
else:
    position = 0.60  # Standard allocation
```